In [1]:
%run run_all_subjects.py --seed 999 --adabn --fnirs-temporal-align egta --eeg-band-mode safe

[SEED CONTROL] Using seed=999 (numpy/torch/python random all seeded, cuDNN deterministic)
NOTE: tsfnet_model package not found; loading definitions from E:\kntu\Thesis\codes\HOPE-LAST-CODE-FILE\new-tsfnet\model\D1\tsfnet_model_notebook.py instead.
[RESULTS] Per-subject session-level JSON/checkpoints for this run go to: E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_b5e27ee74d_seed999
===== CrashSafeLogger started [11SEG_BASELINE_PAPER_MEAN_NOCLIP_NOCLASSWEIGHT_NOSMOTE_GAP_CAFE64_NOASYM_ADABN_EGTA] 2026-09-22 22:17:47 -> E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_b5e27ee74d_seed999\baseline_run_log_20260922_221747_seed999_b5e27ee74d.txt =====
===== FULL BASELINE RUN started 20260922_221747_seed999_b5e27ee74d (seed=999) =====
[ABLATION] asymmetric_pairs=None weight=2.0 ordinal_reg_lambda=0.0 (priority-A1 isolated variable) | lambda_eeg=0.2 lambda_fnirs=0.2 (priority-A3 isolated variable, paper default 0.2/0.2) | cafe_d_model=64 (priority-4 result: reject

In [2]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [3]:
%run run_all_subjects.py --seed 42 --adabn --fnirs-temporal-align egta --eeg-band-mode safe

[SEED CONTROL] Using seed=42 (numpy/torch/python random all seeded, cuDNN deterministic)
NOTE: tsfnet_model package not found; loading definitions from E:\kntu\Thesis\codes\HOPE-LAST-CODE-FILE\new-tsfnet\model\D1\tsfnet_model_notebook.py instead.
[RESULTS] Per-subject session-level JSON/checkpoints for this run go to: E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_b5e27ee74d_seed42
===== CrashSafeLogger started [11SEG_BASELINE_PAPER_MEAN_NOCLIP_NOCLASSWEIGHT_NOSMOTE_GAP_CAFE64_NOASYM_ADABN_EGTA] 2026-09-23 02:19:25 -> E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_b5e27ee74d_seed42\baseline_run_log_20260923_021925_seed42_b5e27ee74d.txt =====
===== FULL BASELINE RUN started 20260923_021925_seed42_b5e27ee74d (seed=42) =====
[ABLATION] asymmetric_pairs=None weight=2.0 ordinal_reg_lambda=0.0 (priority-A1 isolated variable) | lambda_eeg=0.2 lambda_fnirs=0.2 (priority-A3 isolated variable, paper default 0.2/0.2) | cafe_d_model=64 (priority-4 result: rejected) | 

In [4]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [5]:
%run run_all_subjects.py --seed 2024 --adabn --fnirs-temporal-align egta --eeg-band-mode safe

[SEED CONTROL] Using seed=2024 (numpy/torch/python random all seeded, cuDNN deterministic)
NOTE: tsfnet_model package not found; loading definitions from E:\kntu\Thesis\codes\HOPE-LAST-CODE-FILE\new-tsfnet\model\D1\tsfnet_model_notebook.py instead.
[RESULTS] Per-subject session-level JSON/checkpoints for this run go to: E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_b5e27ee74d_seed2024
===== CrashSafeLogger started [11SEG_BASELINE_PAPER_MEAN_NOCLIP_NOCLASSWEIGHT_NOSMOTE_GAP_CAFE64_NOASYM_ADABN_EGTA] 2026-09-23 06:07:44 -> E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_b5e27ee74d_seed2024\baseline_run_log_20260923_060744_seed2024_b5e27ee74d.txt =====
===== FULL BASELINE RUN started 20260923_060744_seed2024_b5e27ee74d (seed=2024) =====
[ABLATION] asymmetric_pairs=None weight=2.0 ordinal_reg_lambda=0.0 (priority-A1 isolated variable) | lambda_eeg=0.2 lambda_fnirs=0.2 (priority-A3 isolated variable, paper default 0.2/0.2) | cafe_d_model=64 (priority-4 result: 

In [7]:
# این سلول را در همان نوت‌بوک/کرنلی بزن که run_all_subjects.py را ران می‌گیری
# (یعنی از قبل tsfnet_model_notebook.py لود شده و TSFNet در دسترس است)

model = TSFNet(num_classes=3, eeg_band_mode="safe")
print("eeg_band_branch exists:", hasattr(model, "eeg_band_branch"))
if hasattr(model, "eeg_band_branch"):
    n_params = sum(p.numel() for p in model.eeg_band_branch.parameters())
    print("eeg_band_branch param count:", n_params)

eeg_band_branch exists: True
eeg_band_branch param count: 5477


In [9]:
path = r"E:\kntu\Thesis\codes\HOPE-LAST-CODE-FILE\new-tsfnet\model\D1\tsfnet_model_notebook.py"
with open(path, encoding="utf-8") as f:
    content = f.read()

# پیدا کردن و چاپ خودِ تابع
start = content.index("def _collect_gate_values")
end = content.index("\ndef ", start + 10)  # تابع بعدی
print(content[start:end])

def _collect_gate_values(model):
    """
    Generically finds every sigmoid-gated scalar parameter on the model
    (the pattern this project uses repeatedly: beta_labram, eeg_band_branch's
    gate, w_labram, ...) and returns their CURRENT sigmoid-activated values
    as a plain dict, for logging into history without needing to know in
    advance which gates a given run's configuration actually constructed.
    Pure read-only inspection -- no gradient, no side effect on training.
    """
    gates = {}
    with torch.no_grad():
        if hasattr(model, "beta_labram"):
            gates["beta_labram"] = torch.sigmoid(model.beta_labram).item()
        if hasattr(model, "w_labram"):
            gates["w_labram"] = torch.sigmoid(model.w_labram).item()
        if hasattr(model, "eeg_band_branch") and hasattr(model.eeg_band_branch, "gate"):
            gates["eeg_band_gate"] = torch.sigmoid(model.eeg_band_branch.gate).item()
        if hasattr(model, "w_eeg"):
            gates["w_eeg_fu

In [10]:
print("eeg_band_gate" in content, "→ اگر False بود، دقیقاً همان چیزی‌ست که حدس زدم")

True → اگر False بود، دقیقاً همان چیزی‌ست که حدس زدم


In [11]:
with open(r"E:\kntu\Thesis\codes\HOPE-LAST-CODE-FILE\new-tsfnet\model\D1\tsfnet_model_notebook.py", encoding="utf-8") as f:
    content = f.read()

import re
for m in re.finditer(r"TSFNet\(", content):
    start = m.start()
    snippet = content[start:start+300]
    has_band = "eeg_band_mode" in snippet
    line_no = content[:start].count("\n") + 1
    print(f"line {line_no}: eeg_band_mode passed = {has_band}")
    print(snippet[:150].replace(chr(10), ' | '))
    print("---")

line 1003: eeg_band_mode passed = False
TSFNet(nn.Module): |     """ |     num_classes=3 (0-/2-/3-back) -- the paper's original tasks (MI, MA, WG) |     were each binary; the classifier heads and 
---
line 3135: eeg_band_mode passed = False
TSFNet(num_classes=3, fusion_combination=fusion_combination, |                         fnirs_temporal_align=fnirs_temporal_align, |                       
---


In [12]:
start = content.index("TSFNet(num_classes=3, fusion_combination=fusion_combination")
# پیدا کردن پرانتز بسته‌ی متناظر (نه فقط ۳۰۰ کاراکتر بعدی)
depth = 0
end = start
for i, ch in enumerate(content[start:], start=start):
    if ch == "(":
        depth += 1
    elif ch == ")":
        depth -= 1
        if depth == 0:
            end = i + 1
            break

full_call = content[start:end]
print(full_call)
print()
print("eeg_band_mode در این فراخوانی هست؟", "eeg_band_mode" in full_call)

TSFNet(num_classes=3, fusion_combination=fusion_combination,
                        fnirs_temporal_align=fnirs_temporal_align,
                        eeg_temporal_mode=eeg_temporal_mode, eeg_lstm_hidden=eeg_lstm_hidden,
                        eeg_readout_mode=eeg_readout_mode, cafe_d_model=cafe_d_model,
                        fnirs_num_segments=fnirs_num_segments,
                        fnirs_chroma_mode=fnirs_chroma_mode,
                        use_fc_features=use_fc_features, fc_num_channels=fc_num_channels,
                        fc_embed_dim=fc_embed_dim,
                        dropout=dropout, cafe_dropout=cafe_dropout,
                        fnirs_num_chromophores=fnirs_num_chromophores,
                        eeg_spatial_mode=eeg_spatial_mode,
                        eeg_spatial_layers=eeg_spatial_layers,
                        eeg_spatial_heads=eeg_spatial_heads,
                        eeg_backbone_mode=eeg_backbone,
                        eeg_band_mode=eeg_band_mo

In [14]:
path2 = r"E:\kntu\Thesis\codes\HOPE-LAST-CODE-FILE\new-tsfnet\model\D1\run_all_subjects.py"
with open(path2, encoding="utf-8") as f:
    content2 = f.read()

start = content2.index("run_subject_cross_session(")
depth = 0
end = start
for i, ch in enumerate(content2[start:], start=start):
    if ch == "(":
        depth += 1
    elif ch == ")":
        depth -= 1
        if depth == 0:
            end = i + 1
            break

full_call = content2[start:end]
print(full_call)
print()
print("eeg_band_mode در این فراخوانی هست؟", "eeg_band_mode" in full_call)

run_subject_cross_session(
                subject_dir, subject_id,
                device="cuda" if torch.cuda.is_available() else "cpu",
                batch_size=16,
                run_diagnostics_flag=RUN_DIAGNOSTICS,
                results_dir=RESULTS_DIR_VARIANT,  # variant+seed-tagged folder -- never overwrites another run's session-level JSON/checkpoints
                fusion_combination=FUSION_COMBINATION,   # priority-2 result: accepted, fixed at "paper_mean"
                class_weighting_scheme=CLASS_WEIGHTING_SCHEME,  # ISOLATED ABLATION (priority-3): only this differs vs. the accepted priority-1+2 baseline
                effective_number_beta=EFFECTIVE_NUMBER_BETA,    # only used if scheme=="effective_number"
                focal_gamma=FOCAL_GAMMA,                        # only used if scheme=="focal"
                oversampling_scheme=OVERSAMPLING_SCHEME,        # ISOLATED ABLATION (priority-3d): data-level axis, independent of class_weighting_scheme
            

In [16]:
start = content.index("def run_subject_cross_session")
end = content.index(":", start)  # تا انتهای خط امضای تابع (قبل از docstring)
print(content[start:end+1])

def run_subject_cross_session(subject_dir, subject_id, device="cuda" if torch.cuda.is_available() else "cpu",
                               batch_size=16, run_diagnostics_flag=False, diagnostics_dir=None,
                               results_dir=None, fusion_combination="paper_mean",
                               class_weighting_scheme="none", effective_number_beta=0.999,
                               focal_gamma=2.0, oversampling_scheme="none", smote_k_neighbors=5,
                               eeg_temporal_mode="gap", eeg_lstm_hidden=64, eeg_readout_mode="h_n",
                               cafe_d_model=64, fnirs_num_segments=21,
                               asymmetric_hard_pairs=None, asymmetric_hard_weight=2.0,
                               ordinal_reg_lambda=0.0, class_count_level="window",
                               fnirs_chroma_mode="both", mhs_lambda=0.0, label_smoothing=0.0,
                               domain_adapt_lambda=0.0,
                               us